# 04 — Register & Deploy SageMaker Endpoint (MAESTRO)

**Run this on MAESTRO SageMaker JupyterLab.**

This notebook has two phases:

### Phase 1 — Model Registration (Data Scientist)
Cells 1–6: Package model artifacts, register in SageMaker Model Registry
with `PendingManualApproval` status. **You run this.**

### Phase 2 — Approval & Deployment (Tech Lead)
Your tech lead approves the model in the MAESTRO UI, then either:
- MAESTRO auto-deploys (if your project has a deploy pipeline), **or**
- Tech lead runs cells 7–12 to deploy manually

Prerequisites:
- Run `02_run_indexing.ipynb` first (FAISS index must exist in S3)
- SageMaker Project created in MAESTRO

# ---
# PHASE 1 — Model Registration (Data Scientist)
# ---

## 1. Setup & Configuration

In [ ]:
import os
import sys
import json
import time
from datetime import datetime

import boto3
import sagemaker

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# SageMaker SDK client (uses MAESTRO config from lcc-init-script)
sagemaker_client = boto3.client('sagemaker', region_name='ap-southeast-1')

# Pre-populated vars from MAESTRO
ASSUME_ROLE = sagemaker.get_execution_role()
S3_BUCKET = sagemaker.session.Session().default_bucket()

# Domain ID
with open('/opt/ml/metadata/resource-metadata.json') as f:
    metadata = json.load(f)
    DOMAIN_ID = metadata.get('DomainId')

# ======================================================================
# YOUR SAGEMAKER PROJECT — update these to match your MAESTRO project
# ======================================================================
SAGEMAKER_PROJECT_ID = 'p-88q9jscqrh0h'
SAGEMAKER_PROJECT_NAME = 'gst-registrants'

# MAESTRO platform vars — DO NOT CHANGE
ENV = 'prod'
AWS_REGION = 'ap-southeast-1'
AWS_ACCOUNT_ID = '819382625810'
SAGEMAKER_PROJECT_NAME_ID = f'{SAGEMAKER_PROJECT_NAME}-{SAGEMAKER_PROJECT_ID}'

# Endpoint settings (used in Phase 2)
INSTANCE_TYPE = 'ml.m5.xlarge'  # 16GB RAM — FAISS index (~7.4GB) needs headroom
INSTANCE_COUNT = 1

# Version tag for resource naming (increment when redeploying)
VERSION_TAG = 'v1'

print(f'Role: {ASSUME_ROLE}')
print(f'S3 Bucket: {S3_BUCKET}')
print(f'Domain ID: {DOMAIN_ID}')
print(f'Project: {SAGEMAKER_PROJECT_NAME} ({SAGEMAKER_PROJECT_ID})')
print(f'Version: {VERSION_TAG}')

## 2. Package model and upload to S3

Create `model.tar.gz` with the inference code and upload to S3.

In [ ]:
os.chdir(PROJECT_ROOT)

from endpoint.package_model import package_model

package_model()

from config import S3_BUCKET as GST_BUCKET, S3_EMBEDDINGS_PREFIX
MODEL_DATA_URL = f's3://{GST_BUCKET}/{S3_EMBEDDINGS_PREFIX}model.tar.gz'
print(f'Model artifact: {MODEL_DATA_URL}')

## 3. Get container image URI

Retrieve the PyTorch inference container image URI — needed for the Model Registry entry.

In [ ]:
image_uri = sagemaker.image_uris.retrieve(
    framework='pytorch',
    region=AWS_REGION,
    version='2.0',
    py_version='py310',
    instance_type=INSTANCE_TYPE,
    image_scope='inference',
)
print(f'Image URI: {image_uri}')

## 4. Create Model Package Group

A Model Package Group holds versioned model packages. We create one per project.

In [ ]:
MODEL_PACKAGE_GROUP_NAME = f'{SAGEMAKER_PROJECT_NAME_ID}'

print('=' * 80)
print('CREATING MODEL PACKAGE GROUP')
print('=' * 80)

try:
    response = sagemaker_client.create_model_package_group(
        ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,
        ModelPackageGroupDescription='GST Entity Matcher — FAISS index for entity name matching',
        Tags=[
            {'Key': 'sagemaker:project-name', 'Value': SAGEMAKER_PROJECT_NAME},
            {'Key': 'sagemaker:project-id', 'Value': SAGEMAKER_PROJECT_ID},
        ],
    )
    print(f'\n✓ Model Package Group created: {response["ModelPackageGroupArn"]}')
except sagemaker_client.exceptions.ClientError as e:
    if 'already exists' in str(e).lower() or e.response['Error']['Code'] == 'ValidationException':
        print(f'✓ Model Package Group "{MODEL_PACKAGE_GROUP_NAME}" already exists, continuing...')
    else:
        raise

## 5. Register Model Package (PendingManualApproval)

Register the model.tar.gz as a versioned Model Package with `PendingManualApproval` status.

**This is where Phase 1 ends.** After this cell, ask your tech lead to approve
the model in the MAESTRO UI:
- Go to **SageMaker Studio → Model Registry → gst-registrants-p-88q9jscqrh0h**
- Select the latest version → **Update status → Approved**

In [ ]:
print('=' * 80)
print('REGISTERING MODEL PACKAGE')
print('=' * 80)

timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')

response = sagemaker_client.create_model_package(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,
    ModelPackageDescription=f'GST Entity Matcher {VERSION_TAG} ({timestamp})',
    InferenceSpecification={
        'Containers': [
            {
                'Image': image_uri,
                'ModelDataUrl': MODEL_DATA_URL,
                'Environment': {
                    'OPENAI_API_KEY': os.getenv('OPENAI_API_KEY', ''),
                    'OPENAI_API_BASE': os.getenv('OPENAI_API_BASE', ''),
                    'SAGEMAKER_TS_RESPONSE_TIMEOUT': '600',
                    'SAGEMAKER_MODEL_SERVER_WORKERS': '1',
                },
            }
        ],
        'SupportedContentTypes': ['application/json'],
        'SupportedResponseMIMETypes': ['application/json', 'text/csv'],
        'SupportedRealtimeInferenceInstanceTypes': [INSTANCE_TYPE],
    },
    ModelApprovalStatus='PendingManualApproval',
)

MODEL_PACKAGE_ARN = response['ModelPackageArn']
print(f'\n✓ Model Package registered: {MODEL_PACKAGE_ARN}')
print(f'  Status: PendingManualApproval')
print(f'\n{"=" * 80}')
print('PHASE 1 COMPLETE — Data Scientist work is done!')
print('=' * 80)
print(f'\nNext step: Ask your tech lead to approve this model in the MAESTRO UI:')
print(f'  Model Registry → {MODEL_PACKAGE_GROUP_NAME} → Approve latest version')
print(f'\nIf MAESTRO does not auto-deploy after approval,')
print(f'the tech lead can run Phase 2 (cells below) to deploy manually.')

# ---
# PHASE 2 — Deployment (Tech Lead / MLOps Engineer)
# ---

Only run the cells below if MAESTRO does not auto-deploy after model approval.
Your tech lead (with MLOps Engineer permissions) runs these cells.

## 6. Get approved Model Package ARN

In [ ]:
# List model packages and find the latest Approved one
response = sagemaker_client.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,
    ModelApprovalStatus='Approved',
    SortBy='CreationTime',
    SortOrder='Descending',
    MaxResults=1,
)

if not response['ModelPackageSummaryList']:
    raise RuntimeError(
        f'No approved model packages found in "{MODEL_PACKAGE_GROUP_NAME}". '
        'Ask the tech lead to approve a model version in the MAESTRO UI first.'
    )

MODEL_PACKAGE_ARN = response['ModelPackageSummaryList'][0]['ModelPackageArn']
print(f'Latest approved Model Package: {MODEL_PACKAGE_ARN}')

# Get the full details (container image, model data URL)
pkg = sagemaker_client.describe_model_package(ModelPackageName=MODEL_PACKAGE_ARN)
container = pkg['InferenceSpecification']['Containers'][0]
print(f'  Image: {container["Image"]}')
print(f'  Model Data: {container["ModelDataUrl"]}')

## 7. Get domain metadata (VPC, KMS)

In [ ]:
response = sagemaker_client.describe_domain(DomainId=DOMAIN_ID)

SUBNET_ID_1 = response['SubnetIds'][0]
SUBNET_ID_2 = response['SubnetIds'][1]
SECURITY_GROUP_ID = response['DefaultUserSettings']['SecurityGroups'][0]

kms_client = boto3.client('kms', region_name=AWS_REGION)
kms_key_alias = f'alias/cmk-agml-{ENV}-domain-{DOMAIN_ID}'
kms_response = kms_client.describe_key(KeyId=kms_key_alias)
KMS_KEY_ARN = kms_response['KeyMetadata']['Arn']

print(f'Subnet 1: {SUBNET_ID_1}')
print(f'Subnet 2: {SUBNET_ID_2}')
print(f'Security Group: {SECURITY_GROUP_ID}')
print(f'KMS Key ARN: {KMS_KEY_ARN}')

## 8. Create SageMaker Model, Endpoint Config & Endpoint

In [ ]:
timestamp = int(datetime.now().timestamp())
MODEL_NAME = f'gst-entity-matcher-{VERSION_TAG}-{timestamp}'
CONFIG_NAME = f'gst-entity-matcher-config-{VERSION_TAG}-{timestamp}'
ENDPOINT_NAME = f'gst-entity-matcher-{VERSION_TAG}'
SAMPLING_PERCENTAGE = 100
DATA_CAPTURE_UPLOAD_PATH = f's3://{S3_BUCKET}/capture/'

print('=' * 80)
print('DEPLOYING FROM APPROVED MODEL PACKAGE')
print('=' * 80)
print(f'  Model Package: {MODEL_PACKAGE_ARN}')
print(f'  Model Name:    {MODEL_NAME}')
print(f'  Config Name:   {CONFIG_NAME}')
print(f'  Endpoint Name: {ENDPOINT_NAME}')

# --- Create Model (from ModelPackage) ---
print(f'\n[1/3] Creating model...')
response = sagemaker_client.create_model(
    ModelName=MODEL_NAME,
    PrimaryContainer={
        'ModelPackageName': MODEL_PACKAGE_ARN,
    },
    ExecutionRoleArn=ASSUME_ROLE,
    # NetworkIsolation must be False — endpoint needs outbound access
    # to MAESTRO's LiteLLM proxy for embedding queries at inference time.
    EnableNetworkIsolation=False,
    VpcConfig={
        'SecurityGroupIds': [SECURITY_GROUP_ID],
        'Subnets': [SUBNET_ID_1, SUBNET_ID_2],
    },
    Tags=[
        {'Key': 'sagemaker:project-name', 'Value': SAGEMAKER_PROJECT_NAME},
        {'Key': 'sagemaker:project-id', 'Value': SAGEMAKER_PROJECT_ID},
    ],
)
print(f'  ✓ Model created: {response["ModelArn"]}')

# --- Create Endpoint Config ---
print(f'\n[2/3] Creating endpoint config...')
response = sagemaker_client.create_endpoint_config(
    EndpointConfigName=CONFIG_NAME,
    ProductionVariants=[
        {
            'VariantName': 'AllTraffic',
            'ModelName': MODEL_NAME,
            'InstanceType': INSTANCE_TYPE,
            'InitialInstanceCount': INSTANCE_COUNT,
            'InitialVariantWeight': 1.0,
        }
    ],
    DataCaptureConfig={
        'EnableCapture': True,
        'InitialSamplingPercentage': SAMPLING_PERCENTAGE,
        'DestinationS3Uri': DATA_CAPTURE_UPLOAD_PATH,
        'CaptureOptions': [
            {'CaptureMode': 'Input'},
            {'CaptureMode': 'Output'},
        ],
        'CaptureContentTypeHeader': {
            'JsonContentTypes': ['application/json'],
        },
        'KmsKeyId': KMS_KEY_ARN,
    },
    KmsKeyId=KMS_KEY_ARN,
    Tags=[
        {'Key': 'sagemaker:project-name', 'Value': SAGEMAKER_PROJECT_NAME},
        {'Key': 'sagemaker:project-id', 'Value': SAGEMAKER_PROJECT_ID},
        {'Key': 'sagemaker:domain-arn', 'Value': f'arn:aws:sagemaker:{AWS_REGION}:{AWS_ACCOUNT_ID}:domain/{DOMAIN_ID}'},
    ],
)
print(f'  ✓ Endpoint config created: {response["EndpointConfigArn"]}')

# --- Create or Update Endpoint ---
print(f'\n[3/3] Creating/updating endpoint...')
endpoint_exists = False
try:
    sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
    endpoint_exists = True
except sagemaker_client.exceptions.ClientError:
    pass

if endpoint_exists:
    sagemaker_client.update_endpoint(
        EndpointName=ENDPOINT_NAME,
        EndpointConfigName=CONFIG_NAME,
    )
    print(f'  ✓ Endpoint update initiated')
else:
    sagemaker_client.create_endpoint(
        EndpointName=ENDPOINT_NAME,
        EndpointConfigName=CONFIG_NAME,
        Tags=[
            {'Key': 'sagemaker:project-name', 'Value': SAGEMAKER_PROJECT_NAME},
            {'Key': 'sagemaker:project-id', 'Value': SAGEMAKER_PROJECT_ID},
            {'Key': 'sagemaker:domain-arn', 'Value': f'arn:aws:sagemaker:{AWS_REGION}:{AWS_ACCOUNT_ID}:domain/{DOMAIN_ID}'},
        ],
    )
    print(f'  ✓ Endpoint creation initiated')

## 9. Wait for Endpoint to be InService

In [ ]:
TIMEOUT = 1800  # 30 minutes
ELAPSED = 0

print(f'Monitoring endpoint: {ENDPOINT_NAME}')
print('-' * 80)

while ELAPSED < TIMEOUT:
    try:
        ep = sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
        status = ep['EndpointStatus']
        ts = datetime.now().strftime('%H:%M:%S')
        print(f'[{ts}] Status: {status} | Elapsed: {ELAPSED}s')

        if status == 'InService':
            print('-' * 80)
            print('✓ Endpoint is now in service!')
            break
        elif status == 'Failed':
            print('-' * 80)
            print(f'✗ Failed: {ep.get("FailureReason", "unknown")}')
            break

        time.sleep(30)
        ELAPSED += 30
    except Exception as e:
        print(f'Error: {e}')
        break

if ELAPSED >= TIMEOUT:
    print('Timeout — endpoint may still be creating in the background')

## 10. Test the endpoint

In [ ]:
runtime = boto3.client('sagemaker-runtime', region_name=AWS_REGION)

test_payload = json.dumps({'entity_names': ['DBS BANK', 'SINGAPORE AIRLINES']})

response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType='application/json',
    Accept='application/json',
    Body=test_payload,
)

results = json.loads(response['Body'].read().decode('utf-8'))
print(json.dumps(results, indent=2))

## 11. Next steps

Now that the endpoint is InService:

1. **Attach to API Gateway:** MAESTRO UI → Domains Pane → Domain Details → API Gateway → attach this endpoint
2. **Get the API URL and API Key** from the API Gateway page
3. **Set env vars on Airbase:**
   - `SAGEMAKER_ENDPOINT_URL` = the API Gateway URL
   - `SAGEMAKER_API_KEY` = the API key
4. **Deploy the Streamlit app** to Airbase

## Cleanup (optional)

Uncomment to delete resources and stop incurring costs.

In [ ]:
# Uncomment to delete
# sagemaker_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
# sagemaker_client.delete_endpoint_config(EndpointConfigName=CONFIG_NAME)
# sagemaker_client.delete_model(ModelName=MODEL_NAME)
# print('Deleted endpoint, config, and model.')